In [ ]:
%pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm


In [ ]:
%pip install bs4 nltk

In [ ]:
from bs4 import BeautifulSoup
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import spacy

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

nltk.download()

nltk.download('punkt')
nltk.download('stopwords')

# Email preprocessing function
def preprocess_email(email_body):
    # Strip HTML tags
    soup = BeautifulSoup(email_body, "html.parser")
    plain_text = soup.get_text()

    # Remove non-alphanumeric characters
    text = re.sub(r"[^\w\s]", "", plain_text)

    # Lowercasing
    text = text.lower()

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words("english"))
    filtered_tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    doc = nlp(" ".join(filtered_tokens))
    lemmatized_tokens = [token.lemma_ for token in doc]

    return text

# Sample email
email_body = """
    <html>
    <body>
    <p>Dear John,</p>
    <p>Here's the confidential budget reporting for Q1 2024. Please review it.</p>
    <p>Best regards,<br>Jane Doe</p>
    </body>
    </html>
"""
email_body_2= """What is your pricing policy and are you available for a quick call this friday"""
processed_email = preprocess_email(email_body)
print(processed_email)


In [ ]:
processed_email_2 = preprocess_email(email_body_2)


In [ ]:
processed_email_2

In [ ]:
print(processed_email)

In [5]:
def redact_pii(text):
    # Process the text with spaCy
    doc = nlp(text)
    
    # Entities considered as PII
    pii_entities = {"PERSON", "GPE", "LOC", "ORG", "EMAIL", "PHONE"}
    
    redacted_text = text
    for ent in doc.ents:
        if ent.label_ in pii_entities:
            # Replace PII entity with a redacted label
            redacted_text = redacted_text.replace(ent.text, "[REDACTED]")
    
    return redacted_text


In [ ]:
redact_pii(email_body_2)

In [ ]:
text = """
John Doe lives in New York and works at Google. His email is johndoe@example.com and phone number is 123-456-7890.
"""

redacted_text = redact_pii(text)
print(redacted_text)


In [ ]:
%pip install presidio-analyzer presidio-anonymizer


In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# Initialize the Presidio analyzer and anonymizer
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

text = "John Doe lives in New York. His email is johndoe@example.com and phone number is 123-456-7890."

# Analyze the text for PII
results = analyzer.analyze(text=text, language="en")

print(results)
# Redact PII from the text
anonymized_text = anonymizer.anonymize(text=text, analyzer_results=results)
print(anonymized_text.text)


In [ ]:
%pip install spacy-experimental


In [ ]:
%pip install https://github.com/explosion/spacy-experimental/releases/download/v0.6.0/en_coreference_web_trf-3.4.0a0-py3-none-any.whl#egg=en_coreference_web_trf


In [3]:
nlp = spacy.load("en_coreference_web_trf")

In [ ]:
doc = nlp("The cats were startled by the dog as it growled at them.")


In [ ]:
doc.spans

In [7]:
def resolve_coreferences_with_spacy_experimental(text):
    
    nlp = spacy.load("en_coreference_web_trf")
    doc = nlp(text)

    # Generate the resolved text
    if doc._.coref_chains:
        resolved_text = doc._.coref_chains.resolve(text)
        return resolved_text
    else:
        return text  # Return the original text if no coreferences are found


In [8]:
text= """Although he was very busy with his work, Peter had had enough of it. He and his wife decided they needed a holiday. They travelled to Spain because they loved the country very much."""

In [ ]:
resolved_text= resolve_coreferences_with_spacy_experimental(text)

In [ ]:
print(resolved_text)

In [ ]:
import coreferee, spacy
nlp = spacy.load("en_coreference_web_trf")
nlp.add_pipe('coreferee')
doc = nlp('Although he was very busy with his work, Peter had had enough of it. He and his wife decided they needed a holiday. They travelled to Spain because they loved the country very much.')
doc._.coref_chains.print()
# Output:
#
# 0: he(1), his(6), Peter(9), He(16), his(18)
# 1: work(7), it(14)
# 2: [He(16); wife(19)], they(21), They(26), they(31)
# 3: Spain(29), country(34)
#
print(doc._.coref_chains.resolve(doc[31]))

In [ ]:
import coreferee

# Load spaCy model and add coreferee
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("coreferee")

# Example text
text = """Alice likes her dog. It's name is tommy. She takes it for a walk every day.
          """
doc = nlp(text)

print(doc._.coref_chains.print())

# Check coreferences
# print(doc._.coref_chains.resolve(text))
print(doc)

In [ ]:
print(doc._.coref_chains.resolve(text))


In [23]:
%pip install crosslingual-coreference --no-deps


Note: you may need to restart the kernel to use updated packages.


In [25]:
pip install crosslingual-coreference


  Using cached allennlp-2.9.3-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp_models-2.9.3-py3-none-any.whl.metadata (23 kB)
  Using cached cached_path-1.1.2-py3-none-any.whl.metadata (6.0 kB)
  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
  Using cached scipy-1.14.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached spacy-3.1.7-cp311-cp311-win_amd64.whl.metadata (18 kB)
  Using cached tqdm-4.64.1-py2.py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.6.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached boto3-1.35.76-py3-none-any.whl.metadata (6.7 kB)
  Using cached google_cloud_storage-2.19.0-py2.py3-none-any.whl.metadata (9.1 kB)
  Using cached huggingface_hub-0.5.1-py3-none-any.whl.metadata (7.1 kB)
INFO: pip is looking at multiple versions of allennlp to determine which version is compatible with other requirements. This could take a while.
  Using cached allennlp-2.9.2-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp-2.

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [143 lines of output]
      running egg_info
      creating C:\Users\aspire-lite\AppData\Local\Temp\pip-pip-egg-info-iofzh31h\checklist.egg-info
      writing C:\Users\aspire-lite\AppData\Local\Temp\pip-pip-egg-info-iofzh31h\checklist.egg-info\PKG-INFO
      writing dependency_links to C:\Users\aspire-lite\AppData\Local\Temp\pip-pip-egg-info-iofzh31h\checklist.egg-info\dependency_links.txt
      writing requirements to C:\Users\aspire-lite\AppData\Local\Temp\pip-pip-egg-info-iofzh31h\checklist.egg-info\requires.txt
      writing top-level names to C:\Users\aspire-lite\AppData\Local\Temp\pip-pip-egg-info-iofzh31h\checklist.egg-info\top_level.txt
      writing manifest file 'C:\Users\aspire-lite\AppData\Local\Temp\pip-pip-egg-info-iofzh31h\checklist.egg-info\SOURCES.txt'
      reading manifest file 'C:\Users\aspire-lite\AppData\Local\Temp\pip-pip-egg-info-iofzh31h\checkl

In [26]:
from crosslingual_coreference import Predictor

text = (
    "Do not forget about Momofuku Ando! He created instant noodles in Osaka. At"
    " that location, Nissin was founded. Many students survived by eating these"
    " noodles, but they don't even know him."
)

# choose minilm for speed/memory and info_xlm for accuracy
predictor = Predictor(
    language="en_core_web_sm", device=-1, model_name="minilm"
)

print(predictor.predict(text)["resolved_text"])
print(predictor.pipe([text])[0]["resolved_text"])
# Note you can also get 'cluster_heads' and 'clusters'
# Output
#
# Do not forget about Momofuku Ando!
# Momofuku Ando created instant noodles in Osaka.
# At Osaka, Nissin was founded.
# Many students survived by eating instant noodles,
# but Many students don't even know Momofuku Ando.

[nltk_data] Downloading package omw-1.4 to C:\Users\aspire-
[nltk_data]     lite\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


ModuleNotFoundError: No module named 'allennlp'

In [27]:
%pip install allennlp allennlp-models


  Using cached allennlp-2.10.1-py3-none-any.whl.metadata (21 kB)
  Using cached allennlp_models-2.10.1-py3-none-any.whl.metadata (23 kB)
INFO: pip is looking at multiple versions of allennlp to determine which version is compatible with other requirements. This could take a while.
  Using cached allennlp-2.10.0-py3-none-any.whl.metadata (20 kB)
  Using cached allennlp-2.9.3-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp-2.9.2-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp-2.9.1-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp-2.9.0-py3-none-any.whl.metadata (18 kB)
  Using cached allennlp-2.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached allennlp-2.7.0-py3-none-any.whl.metadata (17 kB)
INFO: pip is still looking at multiple versions of allennlp to determine which version is compatible with other requirements. This could take a while.
  Using cached allennlp-2.6.0-py3-none-any.whl.metadata (17 kB)
  Using cached allennlp-2.5.0-py3-none-any.whl.metadat

  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
        Using cached setuptools-75.6.0-py3-none-any.whl.metadata (6.7 kB)
        Using cached wheel-0.32.3-py2.py3-none-any.whl.metadata (2.1 kB)
        Using cached Cython-3.0.11-cp311-cp311-win_amd64.whl.metadata (3.2 kB)
        Using cached cymem-2.0.10-cp311-cp311-win_amd64.whl.metadata (8.6 kB)
        Using cached preshed-2.0.1.tar.gz (113 kB)
        Preparing metadata (setup.py): started
        Preparing metadata (setup.py): finished with status 'error'
        error: subprocess-exited-with-error
      
        × python setup.py egg_info did not run successfully.
        │ exit code: 1
        ╰─> [6 lines of output]
            Traceback (most recent call last):
              File "<string>", line 2, in <module>
              File "<pip-setuptools-caller>", line 34, in <module>
              File "C:\Users\aspire-li

In [29]:
%pip install spacy
!python -m spacy download en_core_web_sm


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 2.4 MB/s eta 0:00:06
     ---- ----------------------------------- 1.6/12.8 MB 2.9 MB/s eta 0:00:04
     ------ --------------------------------- 2.1/12.8 MB 3.1 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 3.4 MB/s eta 0:00:03
     ------------- -------------------------- 4.2/12.8 MB 3.5 MB/s eta 0:00:03
     ---------------- ----------------------- 5.2/12.8 MB 3.8 MB/s eta 0:00:02
     ------------------ --------------------- 6.0/12.8 MB 3.8 MB/s eta 0:00:02
     --------------------- ------------------ 6.8/12.8 MB 3.8 MB/s eta 0:00:02
     ----------------------- ---------------- 7.6/12.8 MB 3.8 MB/s eta 0:00:02
     ------------------------- -------------- 8.1/12.8 MB 3.7 MB/s eta 0:00:02
     --------------------------- ------------ 8.9/12.8 MB 3.7 MB/

In [30]:
from crosslingual_coreference import Predictor

text = (
    "Do not forget about Momofuku Ando! He created instant noodles in Osaka. At"
    " that location, Nissin was founded. Many students survived by eating these"
    " noodles, but they don't even know him."
)

# choose minilm for speed/memory and info_xlm for accuracy
predictor = Predictor(
    language="en_core_web_sm", device=-1, model_name="minilm"
)

print(predictor.predict(text)["resolved_text"])
print(predictor.pipe([text])[0]["resolved_text"])

[nltk_data] Downloading package omw-1.4 to C:\Users\aspire-
[nltk_data]     lite\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


ModuleNotFoundError: No module named 'allennlp'

In [33]:
!pip install allennlp

  Using cached allennlp-2.10.1-py3-none-any.whl.metadata (21 kB)
INFO: pip is looking at multiple versions of allennlp to determine which version is compatible with other requirements. This could take a while.
  Using cached allennlp-2.10.0-py3-none-any.whl.metadata (20 kB)
  Using cached allennlp-2.9.3-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp-2.9.2-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp-2.9.1-py3-none-any.whl.metadata (19 kB)
  Using cached allennlp-2.9.0-py3-none-any.whl.metadata (18 kB)
  Using cached allennlp-2.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached allennlp-2.7.0-py3-none-any.whl.metadata (17 kB)
INFO: pip is still looking at multiple versions of allennlp to determine which version is compatible with other requirements. This could take a while.
  Using cached allennlp-2.6.0-py3-none-any.whl.metadata (17 kB)
  Using cached allennlp-2.5.0-py3-none-any.whl.metadata (17 kB)
  Using cached allennlp-2.4.0-py3-none-any.whl.metadata (17 kB

  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
        Using cached setuptools-75.6.0-py3-none-any.whl.metadata (6.7 kB)
        Using cached wheel-0.32.3-py2.py3-none-any.whl.metadata (2.1 kB)
        Using cached Cython-3.0.11-cp311-cp311-win_amd64.whl.metadata (3.2 kB)
        Using cached cymem-2.0.10-cp311-cp311-win_amd64.whl.metadata (8.6 kB)
        Using cached preshed-2.0.1.tar.gz (113 kB)
        Preparing metadata (setup.py): started
        Preparing metadata (setup.py): finished with status 'error'
        error: subprocess-exited-with-error
      
        × python setup.py egg_info did not run successfully.
        │ exit code: 1
        ╰─> [6 lines of output]
            Traceback (most recent call last):
              File "<string>", line 2, in <module>
              File "<pip-setuptools-caller>", line 34, in <module>
              File "C:\Users\aspire-li

: 

In [32]:
!pip install preshed

In [ ]:
%pip install --upgrade pip setuptools wheel


In [ ]:
resolved_text= resolve_coreferences_with_crosslingual(text)

In [9]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# Initialize Presidio components
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def anonymize_text_with_mapping(text):
    # Analyze the text to detect PII
    analysis_results = analyzer.analyze(text=text, language="en")

    # Create a mapping for placeholders
    pii_mapping = {}
    unique_id = {}

    for i, entity in enumerate(analysis_results):
        entity_type = entity.entity_type

        # Keep track of unique IDs for each entity type
        if entity_type not in unique_id:
            unique_id[entity_type] = 0
        unique_id[entity_type] += 1

        # Generate a unique placeholder
        placeholder = f"[REDACTED_{entity_type}_{unique_id[entity_type]}]"
        original_value = text[entity.start:entity.end]

        # Add to the mapping
        pii_mapping[placeholder] = original_value

    # Anonymize the text by replacing detected PII with placeholders
    anonymized_text = text
    for placeholder, original_value in pii_mapping.items():
        anonymized_text = anonymized_text.replace(original_value, placeholder)

    return anonymized_text, pii_mapping


In [10]:
text = """
John Doe and Jane Smith are friends. John lives in New York, and Jane lives in California.
Their emails are johndoe@example.com and janesmith@example.com.
"""


In [11]:
anonymized_text, pii_mapping= anonymize_text_with_mapping(text)

In [ ]:
print(anonymized_text)
print(pii_mapping)